## DuckDB Iceberg Sample

This sample shows how to export a DuckDB table as an Iceberg table.

### Setting up

In [ ]:
import os
from pathlib import Path

import duckdb
import pyarrow
from pyiceberg.catalog import NoSuchTableError
from pyiceberg.catalog.sql import SqlCatalog

### Creating catalog

In [ ]:
warehouse_dir_path = Path(".tmp") / "iceberg" / "sample"

warehouse_dir_path.mkdir(
    exist_ok=True,
    parents=True,
)

catalog_options = {
    "uri": f"sqlite:///{warehouse_dir_path}/catalog.db",
    "warehouse": f"file://{warehouse_dir_path}",
}

catalog = SqlCatalog(
    "default",
    **catalog_options,
)

### Creating Iceberg namespace

In [ ]:
catalog.create_namespace_if_not_exists(
    namespace="source__mysql",
    properties={"description": "an sample namespace"},
)

In [ ]:
catalog.list_namespaces(namespace="source__mysql")

In [ ]:
catalog.load_namespace_properties(namespace="source__mysql")

### Reading data from DuckDB database

In [ ]:
db = duckdb.connect(
    database=Path(os.environ["SD_DATA_DIR"]) / "sample.db",
    read_only=True,
)

ar_table = db.sql("FROM source__mysql.users").arrow()

db.close()

### Creating Iceberg table

In [ ]:
# https://github.com/apache/iceberg-python/issues/541
def cast_arrow_table(ar_table):
    fields = []
    for f in ar_table.schema:
        if pyarrow.types.is_timestamp(f.type):
            fields.append(pyarrow.field(f.name, pyarrow.timestamp("us")))
        elif pyarrow.types.is_null(f.type):
            fields.append(pyarrow.field(f.name, pyarrow.string()))
        else:
            fields.append(f)

    return ar_table.cast(pyarrow.schema(fields))


try:
    catalog.drop_table("source__mysql.users")
except NoSuchTableError:
    pass

ar_table = cast_arrow_table(ar_table)

iceberg_table = catalog.create_table(
    identifier="source__mysql.users",
    schema=ar_table.schema,
)

iceberg_table.overwrite(ar_table)

### Reading data from Iceberg table

In [ ]:
iceberg_table.scan().to_pandas()